In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from pathlib import Path

output_dir = Path("neuroscience_project/results")

connectivity_matrix = np.load(
    output_dir / "functional_connectivity.npy"
)

print("Matrix shape:", connectivity_matrix.shape)
print("Data loaded successfully")

C:\Users\praja\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\praja\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Matrix shape: (74, 74)
Data loaded successfully


In [3]:
print("NaN values:", np.isnan(connectivity_matrix).sum())
print("Infinite values:", np.isinf(connectivity_matrix).sum())

print("Symmetric:", np.allclose(
    connectivity_matrix,
    connectivity_matrix.T,
    atol=1e-8
))

print("Diagonal:", np.diag(connectivity_matrix)[:5])

eigenvalues = np.linalg.eigvalsh(connectivity_matrix)

print("Minimum eigenvalue:", eigenvalues.min())

assert np.isfinite(connectivity_matrix).all()
assert np.allclose(connectivity_matrix, connectivity_matrix.T, atol=1e-8)
assert np.allclose(np.diag(connectivity_matrix), 1, atol=1e-8)
assert eigenvalues.min() >= -1e-6

print("Matrix validation passed")


NaN values: 0
Infinite values: 0
Symmetric: True
Diagonal: [1. 1. 1. 1. 1.]
Minimum eigenvalue: 0.00013191945269396351
Matrix validation passed


In [5]:
distance_matrix = np.sqrt(
    np.maximum(0, 2 * (1 - connectivity_matrix))
)

np.fill_diagonal(distance_matrix, 0)

print("Distance matrix shape:", distance_matrix.shape)
print("Minimum distance:", distance_matrix.min())
print("Maximum distance:", distance_matrix.max())
print("Finite values:", np.isfinite(distance_matrix).all())

Distance matrix shape: (74, 74)
Minimum distance: 0.0
Maximum distance: 1.5008524278813047
Finite values: True


In [7]:
upper_distances = distance_matrix[
    np.triu_indices_from(distance_matrix, k=1)
]

max_edge_length = upper_distances.max()

print("Number of brain regions:", distance_matrix.shape[0])
print("Number of pairwise distances:", len(upper_distances))
print("Filtration maximum:", max_edge_length)

print("Vietoris-Rips filtration parameters ready")

Number of brain regions: 74
Number of pairwise distances: 2701
Filtration maximum: 1.5008524278813047
Vietoris-Rips filtration parameters ready


In [ ]:
%pip install ripser

In [ ]:
from ripser import ripser

tda_result = ripser(
    distance_matrix,
    distance_matrix=True,
    maxdim=1,
    thresh=max_edge_length
)

diagrams = tda_result["dgms"]

H0 = diagrams[0]
H1 = diagrams[1]

print("H0 shape:", H0.shape)
print("H1 shape:", H1.shape)
print("Persistent homology calculated")